# Phase 2: Feature Extraction

**Objective:** Extract MFCC features from all MIMII machine types and noise conditions.

**Implementations details:**

- Loops over all 4 machine types (fan, pump, valve, slide_rail)
- Loops over all 3 noise conditions (-6 dB, 0 dB, 6 dB)
- Dynamically discovers sample counts
- Saves datasets for Phase 3 preprocessing

## Setup

In [1]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, 'C:/Users/Letizia/Documents/sound-anomaly-detection/src')
from audio_loader import AudioLoader

print("✓ All imports successful!")
print(f"\nAvailable machine types: {AudioLoader.VALID_MACHINE_TYPES}")
print(f"Available conditions: {AudioLoader.VALID_CONDITIONS}")

✓ All imports successful!

Available machine types: ['fan', 'pump', 'valve', 'slider']
Available conditions: ['-6_dB', '0_dB', '6_dB']


## TEST: Test with Single Condition First

Test with fan + 0_dB to to verify the setup works before running all conditions:

In [2]:
# Quick test: load fan data at 0 dB
print("Testing with fan at 0_dB...\n")

loader = AudioLoader(data_root="C:/Users/Letizia/Documents/sound-anomaly-detection/data", sr=16000, n_mfcc=13, machine_type="fan")

conditions_available = loader.list_available_conditions()
print(f"Available conditions for fan: {conditions_available}")

machine_ids = loader.list_machine_ids('0_dB')
print(f"Available machine IDs at 0_dB: {machine_ids}")

# Load just id_00 to verify
data_test = loader.load_condition_dataset(
    condition='0_dB',
    machine_ids=['id_00'],
    aggregate_method='mean'
)

print(f"\n✓ Test successful!")
print(f"Loaded machine: id_00")
print(f"  Normal clips: {len(data_test['machine_ids']['id_00']['normal']['features'])}")
print(f"  Abnormal clips: {len(data_test['machine_ids']['id_00']['abnormal']['features'])}")

Testing with fan at 0_dB...

✓ AudioLoader initialized
  - Data root: C:\Users\Letizia\Documents\sound-anomaly-detection\data
  - Machine type: fan
  - Sampling rate: 16000 Hz
  - MFCC coefficients: 13
Available conditions for fan: ['-6_dB', '0_dB', '6_dB']
Available machine IDs at 0_dB: ['id_00', 'id_02', 'id_04', 'id_06']

Loading condition: 0_dB (fan)
  Loading id_00 normal sounds... ✓ 1011 clips
  Loading id_00 abnormal sounds... ✓ 407 clips

✓ Test successful!
Loaded machine: id_00
  Normal clips: 1011
  Abnormal clips: 407


## Full Experiment: Extract All Conditions & Machine Types

This loop extracts features for all combinations and saves them.  
**Warning**: 30+ minutes

In [3]:
# Create results directory
results_dir = Path("C:/Users/Letizia/Documents/sound-anomaly-detection/results")
results_dir.mkdir(exist_ok=True)

# Aggregation method: choose 'mean' or 'mean_std'
# 'mean' = 13-d vectors (faster, standard)
# 'mean_std' = 26-d vectors (captures temporal variability, slower)
AGGREGATE_METHOD = 'mean_std'

# Data structure to track what we extract
experiments = {
    'configs': [],  # Track what we did
    'results': {}   # Store extracted data
}

MACHINE_TYPES = ['fan', 'pump', 'slider', 'valve']
CONDITIONS = ['-6_dB', '0_dB', '6_dB']

# Loop over machine types
for machine_type in MACHINE_TYPES:
    print(f"\n{'='*70}")
    print(f"MACHINE TYPE: {machine_type.upper()}")
    print(f"{'='*70}")
    
    loader = AudioLoader(
        data_root="C:/Users/Letizia/Documents/sound-anomaly-detection/data",
        sr=16000,
        n_mfcc=13,
        machine_type=machine_type
    )
    
    # Loop over noise conditions
    for condition in CONDITIONS:
        print(f"\n  Condition: {condition}")
        
        try:
            # Load data for training machines (id_00, id_02) and test machines (id_04, id_06)
            data = loader.load_condition_dataset(
                condition=condition,
                machine_ids=['id_00', 'id_02', 'id_04', 'id_06'],
                aggregate_method=AGGREGATE_METHOD
            )
            
            # Dynamically get sample counts
            n_id00_normal = len(data['machine_ids']['id_00']['normal']['features'])
            n_id02_normal = len(data['machine_ids']['id_02']['normal']['features'])
            
            print(f"\n  Preparing datasets...")
            
            # Training set: normal sounds from id_00 and id_02 only
            X_train = np.vstack([
                np.array(data['machine_ids']['id_00']['normal']['features']),
                np.array(data['machine_ids']['id_02']['normal']['features'])
            ])
            
            # Machine IDs for training (dynamic)
            machine_ids_train = ['id_00'] * n_id00_normal + ['id_02'] * n_id02_normal
            
            # For each test machine (id_04, id_06), create separate test sets
            for test_id in ['id_04', 'id_06']:
                X_test = np.vstack([
                    np.array(data['machine_ids'][test_id]['normal']['features']),
                    np.array(data['machine_ids'][test_id]['abnormal']['features'])
                ])
                
                n_normal = len(data['machine_ids'][test_id]['normal']['features'])
                n_abnormal = len(data['machine_ids'][test_id]['abnormal']['features'])
                
                y_test = np.concatenate([
                    np.zeros(n_normal),   # 0 = normal
                    np.ones(n_abnormal)   # 1 = abnormal
                ])
                
                # Create filename with machine_type and condition
                condition_short = condition.replace('_dB', 'dB')
                filename_train = f"X_train_{condition_short}_{machine_type}.npy"
                filename_test = f"X_test_{condition_short}_{machine_type}_{test_id}.npy"
                filename_labels = f"y_test_{condition_short}_{machine_type}_{test_id}.npy"
                filename_machineids = f"machine_ids_train_{condition_short}_{machine_type}.pkl"
                
                # Save datasets
                np.save(results_dir / filename_train, X_train)
                np.save(results_dir / filename_test, X_test)
                np.save(results_dir / filename_labels, y_test)
                
                # Also save machine_ids as pickle for reference
                import pickle
                with open(results_dir / filename_machineids, 'wb') as f:
                    pickle.dump(machine_ids_train, f)
                
                # Track in experiments dict
                config_key = f"{machine_type}-{condition}-{test_id}"
                experiments['results'][config_key] = {
                    'machine_type': machine_type,
                    'condition': condition,
                    'test_id': test_id,
                    'X_train': X_train.shape,
                    'X_test': X_test.shape,
                    'n_normal': n_normal,
                    'n_abnormal': n_abnormal,
                    'n_train_id00': n_id00_normal,
                    'n_train_id02': n_id02_normal,
                    'files': [filename_train, filename_test, filename_labels]
                }
                
                print(f"    {test_id}: saved (train: {X_train.shape}, test: {X_test.shape})")
        
        except FileNotFoundError as e:
            print(f"    ⚠ Skipping {condition}: {str(e)[:50]}...")
        except Exception as e:
            print(f"    ✗ Error: {e}")

print(f"\n\n{'='*70}")
print("✓ PHASE 2 COMPLETE")
print(f"{'='*70}")
print(f"\nTotal configurations extracted: {len(experiments['results'])}")
print(f"\nFiles saved to: {results_dir.absolute()}")


MACHINE TYPE: FAN
✓ AudioLoader initialized
  - Data root: C:\Users\Letizia\Documents\sound-anomaly-detection\data
  - Machine type: fan
  - Sampling rate: 16000 Hz
  - MFCC coefficients: 13

  Condition: -6_dB

Loading condition: -6_dB (fan)
  Loading id_00 normal sounds... ✓ 1011 clips
  Loading id_00 abnormal sounds... ✓ 407 clips
  Loading id_02 normal sounds... ✓ 1016 clips
  Loading id_02 abnormal sounds... ✓ 359 clips
  Loading id_04 normal sounds... ✓ 1033 clips
  Loading id_04 abnormal sounds... ✓ 348 clips
  Loading id_06 normal sounds... ✓ 1015 clips
  Loading id_06 abnormal sounds... ✓ 361 clips

  Preparing datasets...
    id_04: saved (train: (2027, 26), test: (1381, 26))
    id_06: saved (train: (2027, 26), test: (1376, 26))

  Condition: 0_dB

Loading condition: 0_dB (fan)
  Loading id_00 normal sounds... ✓ 1011 clips
  Loading id_00 abnormal sounds... ✓ 407 clips
  Loading id_02 normal sounds... ✓ 1016 clips
  Loading id_02 abnormal sounds... ✓ 359 clips
  Loading id_

## Summary of Extracted Data

In [4]:
import pandas as pd

summary_data = []
for config_key, info in experiments['results'].items():
    summary_data.append({
        'Machine': info['machine_type'],
        'Condition': info['condition'],
        'Test ID': info['test_id'],
        'Train Shape': info['X_train'],
        'Test Shape': info['X_test'],
        'Normal': info['n_normal'],
        'Abnormal': info['n_abnormal'],
        'Total Test': info['n_normal'] + info['n_abnormal']
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values(['Machine', 'Condition', 'Test ID']).reset_index(drop=True)

print("\n" + "="*70)
print("EXTRACTED DATASETS SUMMARY")
print("="*70)
print(summary_df.to_string(index=False))

print(f"\nTotal datasets: {len(summary_df)}")
print(f"Machine types: {summary_df['Machine'].nunique()} → {sorted(summary_df['Machine'].unique())}")
print(f"Conditions: {summary_df['Condition'].nunique()} → {sorted(summary_df['Condition'].unique())}")
print(f"Test IDs: {summary_df['Test ID'].nunique()} → {sorted(summary_df['Test ID'].unique())}")


EXTRACTED DATASETS SUMMARY
Machine Condition Test ID Train Shape Test Shape  Normal  Abnormal  Total Test
    fan     -6_dB   id_04  (2027, 26) (1381, 26)    1033       348        1381
    fan     -6_dB   id_06  (2027, 26) (1376, 26)    1015       361        1376
    fan      0_dB   id_04  (2027, 26) (1381, 26)    1033       348        1381
    fan      0_dB   id_06  (2027, 26) (1376, 26)    1015       361        1376
    fan      6_dB   id_04  (2027, 26) (1381, 26)    1033       348        1381
    fan      6_dB   id_06  (2027, 26) (1376, 26)    1015       361        1376
   pump     -6_dB   id_04  (2011, 26)  (802, 26)     702       100         802
   pump     -6_dB   id_06  (2011, 26) (1138, 26)    1036       102        1138
   pump      0_dB   id_04  (2011, 26)  (802, 26)     702       100         802
   pump      0_dB   id_06  (2011, 26) (1138, 26)    1036       102        1138
   pump      6_dB   id_04  (2011, 26)  (802, 26)     702       100         802
   pump      6_dB   id_0

## Achieved:
- Feature vectors for all machine types (fan, pump, valve, slide_rail)
- All noise conditions (-6 dB, 0 dB, 6 dB)
- Training sets (id_00 + id_02 normal sounds)
- Test sets for both id_04 and id_06 (normal + abnormal)